# Speculative Decoding with Tree Attention (Medusa & Eagle)

## 1. Linear Speculative Decoding vs Tree Attention
Standard speculative decoding generates a linear chain of $K$ draft tokens from a small draft model. The target LLM evaluates the chain in a single forward pass:

$$\text{Candidate Chain: } [x_1, x_2, x_3, x_4]$$

If the target LLM rejects $x_2$, then $x_3$ and $x_4$ are discarded immediately, yielding an acceptance length of only 1 token.

### Tree Attention Architecture
Modern inference frameworks (Medusa, Eagle, SpecInfer) replace the linear chain with a **tree of candidate branches**. For instance, draft heads predict top-3 candidates for step 1, and each spawns top-2 candidates for step 2, generating a tree of 16-32 potential token paths:

$$\text{Tree Topology: Root } \to \{A_1, A_2, A_3\} \to \dots$$

### The 2D Tree Attention Causal Mask
Because the target LLM processes the entire tree simultaneously in **one single forward pass**, tokens must attend only to their **true topological ancestors** in the tree:

$$M_{i, j} = \begin{cases} 0 & \text{if node } j \text{ is an ancestor of node } i \\ -\infty & \text{otherwise} \end{cases}$$

This non-triangular 2D attention mask ensures strict causality across branching paths without mutual contamination!

In [ ]:
import numpy as np

class TreeAttentionEngine:
    """
    Demonstrates Speculative Decoding Tree topology and ancestor attention mask generation.
    """
    def __init__(self, tree_structure):
        # tree_structure: list of (node_id, parent_node_id, token_id)
        # Root is node 0 with parent -1
        self.tree = tree_structure
        self.num_nodes = len(tree_structure)

    def build_tree_attention_mask(self):
        """
        Builds [num_nodes, num_nodes] causal mask where entry (i, j) is 1
        if and only if j is an ancestor of i (or i == j).
        """
        # Build ancestor map
        ancestors = {i: set([i]) for i in range(self.num_nodes)}
        for node_id, parent_id, _ in self.tree:
            if parent_id != -1:
                ancestors[node_id].update(ancestors[parent_id])
                
        # Fill mask matrix: 0 for allowed attention, -1e9 for masked
        mask = np.full((self.num_nodes, self.num_nodes), -1e9, dtype=np.float32)
        for i in range(self.num_nodes):
            for j in ancestors[i]:
                mask[i, j] = 0.0 # Can attend to ancestors
                
        return mask, ancestors

# Define candidate tree:
# Node 0: Root token
# Nodes 1, 2: Two candidate branches for step 1
# Nodes 3, 4: Children of Node 1
# Nodes 5, 6: Children of Node 2
tree_nodes = [
    (0, -1, "The"),
    (1,  0, "cat"),
    (2,  0, "dog"),
    (3,  1, "sat"),
    (4,  1, "jumped"),
    (5,  2, "barked"),
    (6,  2, "ran")
]

engine = TreeAttentionEngine(tree_nodes)
tree_mask, anc_map = engine.build_tree_attention_mask()

print(f"Tree Size: {len(tree_nodes)} candidate tokens evaluated in a SINGLE target pass!")
print("Tree Attention Adjacency Mask (0.0 = Attends, -inf = Blocked):")
print((tree_mask == 0.0).astype(int))

# Verify causality: Node 3 ("sat") can attend to Node 1 ("cat") and Node 0 ("The"), but NOT Node 2 ("dog")
assert tree_mask[3, 1] == 0.0
assert tree_mask[3, 2] == -1e9
print("
Topological causality verified: Node 3 attending to Node 1 allowed; Node 3 attending to Node 2 blocked!")

## 2. Speedup Comparison: Chain vs Tree Speculative Decoding
- **Linear Chain ($K=5$)**: If average acceptance rate is $p=0.6$, expected accepted tokens per step:
  $$\mathbb{E}[\text{tokens}] = \sum_{i=1}^K p^i \approx 1.5 \text{ to } 1.8\times \text{ speedup}$$
- **Tree Attention (32 Candidates)**: Evaluating multiple parallel branches raises the probability that at least one deep path matches target LLM logits, boosting speedup to **$2.5\times - 3.5\times$**!

## 3. High-Yield Interview Questions & Best Practices

### Q1: How does Medusa generate draft tokens without a separate draft model?
**Answer:** Medusa does not use a secondary small draft model (which incurs separate memory and scheduling overhead). Instead, Medusa attaches **$K$ lightweight MLP prediction heads** directly on top of the target LLM's final hidden state. Head 1 predicts token $t+1$, Head 2 predicts $t+2$, and Head 3 predicts $t+3$. The generated predictions are combined into a tree attention buffer and validated in the very next target LLM forward step.

### Q2: What is the optimal tree topology for tree-based speculative decoding?
**Answer:** The optimal tree depends on the draft acceptance probability distribution. A balanced tree (e.g. 4 branches with 3 sub-branches) works well for open-ended generation, whereas a deep narrow tree (e.g. 2 branches with 6 sub-branches) is optimal for high-predictability code syntax or JSON generation where draft accuracy is high.